# 13차시 손글씨 숫자 맞히기: 신경망 첫 만남
오늘은 **이미지**를 다룹니다. 손으로 쓴 숫자 사진을 보고 0~9 중 무엇인지 맞힙니다.

**오늘 새로 배우는 것**
1. **층(layer)과 노드**: 신경망은 층을 쌓아 만든 모델, 층 안의 계산 단위가 노드
2. **epoch**: 학습 데이터 전체를 한 바퀴 학습하는 것
3. **loss**: 모델이 얼마나 틀렸는지 (작을수록 좋음)
4. **학습 곡선**: epoch마다 loss와 정확도를 이은 선 그래프

위에서부터 순서대로 한 셀씩 실행하세요. 원본 코드 아래 **🔁 변형** 셀은 조건이 바뀌었을 때 숫자만 바꿔 쓰는 용도입니다.

**목차**: A. 공식 정리 / 0. 사진은 숫자표다 / 1. 데이터 준비 / 2. 신경망 모델 / 3. 학습 / 4. 학습 곡선 / 5. 노드 수 비교 / 6. 개념 문제 / 7. 논술형

# A. 공식 정리
### 출력 크기
| 층 | 들어가는 모양 | 나오는 모양 | 규칙 |
|---|---|---|---|
| `Input(shape=(28, 28))` | - | `(28, 28)` | (높이, 너비) |
| `Flatten()` | `(28, 28)` | `(784,)` | **모두 곱하기**: 28 × 28 = 784 |
| `Dense(128)` | `(784,)` | `(128,)` | 노드 수 |
| `Dense(10)` | `(128,)` | `(10,)` | 출력층 노드 수 = **분류할 종류 수** |

### 파라미터 계산
**Dense: (들어오는 수 + 1) × 노드 수** (`+ 1`은 노드마다 더하는 값, 편향)
- Dense(128): (784 + 1) × 128 = **100,480**
- Dense(10): (128 + 1) × 10 = **1,290**
- 합계 **101,770** / `Flatten`, `Input`은 **0**

### 미리 계산해 둔 변형 모델 (직접 확인한 값)
| 모델 | 층별 | 합계 |
|---|---|---|
| 노드 128개 (수업) | 100,480 + 1,290 | **101,770** |
| 노드 16개 (수업) | 12,560 + 170 | **12,730** |
| 노드 64개 | 50,240 + 650 | 50,890 |
| 노드 256개 | 200,960 + 2,570 | 203,530 |
| 은닉층 2개 (128 → 64) | 100,480 + 8,256 + 650 | 109,386 |

# 0. 사진은 숫자표다
컴퓨터에게 사진은 **0~255 숫자가 들어 있는 표(배열)** 입니다. 0은 검은색, 255는 하얀색.

In [ ]:
# 라이브러리 불러오고 예제 사진 파일 만들기
import os                                   # 파일이 있는지 확인하는 도구
import cv2                                  # 사진을 읽고, 크기를 바꾸고, 필터를 적용하는 도구 (OpenCV)
import numpy as np                          # 숫자표(배열)를 만들고 계산하는 도구
import matplotlib.pyplot as plt             # 사진과 그래프를 그리는 도구
from skimage import data                    # 예제 사진(고양이)이 들어 있는 도구
if not os.path.exists('sample.png'):        # sample.png 파일이 아직 없으면 (있으면 덮어쓰지 않음)
    cv2.imwrite('sample.png', cv2.cvtColor(data.chelsea(), cv2.COLOR_RGB2BGR))   # 고양이 사진을 파일로 저장 (cv2는 BGR 순서라서 RGB -> BGR로 바꿈)

In [ ]:
# 사진 파일을 흑백으로 불러와서 보기
filename = 'sample.png'                                 # 불러올 사진 파일 이름 (내 사진이면 'me.jpg' 처럼 바꾸기)
img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)        # 사진을 흑백으로 읽어서 숫자표로 가져옴
print(img is None)                                      # False 이면 정상, True 이면 파일 이름이 틀린 것
plt.imshow(img, cmap='gray')                            # 숫자표를 흑백 사진으로 그리기
plt.axis('off')                                         # 눈금(축) 숨기기
plt.show()                                              # 화면에 보여주기

In [ ]:
# 사진의 크기와 왼쪽 위 5x5 칸의 숫자 보기
print(img.shape)                # (높이, 너비) -> (300, 451) : 높이 300칸, 너비 451칸
print(img[:5, :5])              # [위에서 5줄, 왼쪽에서 5칸] 숫자 보기 (0~255 사이)

#### 🔁 변형 ①: 다른 위치의 숫자 보기 / 가장 밝은 값·어두운 값

In [ ]:
print(img[100:105, 200:205])    # 위에서 100~104줄, 왼쪽에서 200~204칸 (슬라이싱은 끝 숫자 미포함)
print(img[0, 0])                # 맨 왼쪽 위 한 칸(픽셀)의 값
print(img.max(), img.min())     # 가장 밝은 값, 가장 어두운 값

In [ ]:
# 사진을 28x28로 줄이기
small = cv2.resize(img, (28, 28))       # cv2.resize(사진, (너비, 높이)) -> 28x28 로 줄임
print(small.shape)                      # (28, 28) -> 28 x 28 = 784개 숫자짜리 표
plt.imshow(small, cmap='gray')          # 줄인 사진 그리기 (모자이크처럼 보임)
plt.show()                              # 화면에 보여주기

#### 🔁 변형 ②: 가로·세로가 다른 크기로 줄이기 (순서 함정!)
`cv2.resize`는 **(너비, 높이)**, `shape`는 **(높이, 너비)** 로 순서가 반대입니다.

In [ ]:
wide = cv2.resize(img, (40, 20))        # 너비 40, 높이 20 으로 줄이기
print(wide.shape)                       # (20, 40) -> shape 는 (높이, 너비) 순서라 거꾸로 나옴

#### 🔁 변형 ③ ⚠: 컬러로 불러오기 (채널 3)

In [ ]:
color = cv2.imread(filename)                            # IMREAD_GRAYSCALE 을 빼면 컬러로 읽음
print(color.shape)                                      # (300, 451, 3) -> (높이, 너비, 채널) 컬러는 채널 3
color = cv2.cvtColor(color, cv2.COLOR_BGR2RGB)          # cv2는 BGR 순서라서 화면에 맞게 RGB 로 바꿈
plt.imshow(color)                                       # 컬러 사진은 cmap 없이 그림
plt.axis('off')                                         # 눈금 숨기기
plt.show()                                              # 화면에 보여주기

### 해석해 보기
1. `(300, 451)`는 사진이 높이 `300`칸, 너비 `451`칸이라는 뜻이다.
2. 사진 한 칸(픽셀)의 값이 `0`이면 `검은`색, `255`이면 `하얀`색이다.
3. 사진 한 장은 사실 `숫자`로 이루어진 표이다.
4. 28×28 사진은 숫자 `784`개짜리 표이다.

# 1. 데이터 준비

In [ ]:
# 라이브러리 불러오기
from tensorflow.keras.datasets import mnist                     # 손글씨 숫자 사진 7만 장이 든 데이터
from tensorflow.keras.models import Sequential                  # 층을 순서대로 쌓아 모델을 만드는 틀
from tensorflow.keras.layers import Input, Flatten, Dense       # 모델에 쌓을 층들

In [ ]:
# 손글씨 숫자 데이터 불러오기 (학습용 60000장, 테스트용 10000장)
(X_train, y_train), (X_test, y_test) = mnist.load_data()        # (학습용 사진, 학습용 정답), (테스트용 사진, 테스트용 정답)
print(X_train.shape)                                            # (60000, 28, 28) -> (장수, 높이, 너비)
print(y_train[:10])                                             # 앞에서 10개 정답 -> [5 0 4 1 9 2 1 3 1 4]

In [ ]:
# 앞의 5장을 그림과 정답으로 보기
for i in range(5):                                  # 0번부터 4번 사진까지
    plt.subplot(1, 5, i + 1)                        # 1행 5열 중 i+1 번째 칸
    plt.imshow(X_train[i], cmap='gray')             # i번째 사진 흑백으로 그리기
    plt.title(y_train[i])                           # 정답 숫자를 제목으로
    plt.axis('off')                                 # 눈금 숨기기
plt.show()                                          # 보여주기

In [ ]:
# 픽셀값의 최댓값과 최솟값 확인
print(X_train[2].max(), X_train[2].min())           # 255 0 -> 가장 밝은 값, 가장 어두운 값

In [ ]:
# 픽셀값을 0~1로 만들기 (정규화)
X_train = X_train / 255         # 가장 큰 값 255로 나누면 0~255 -> 0~1 (가장 밝은 칸 255 -> 1.0)
X_test = X_test / 255           # 테스트용도 똑같이 나눠야 함 (한쪽만 나누면 안 됨!)
print(X_train.max())            # 1.0

#### 🔁 변형 ④: 옷 사진(fashion_mnist)으로 바꾸기
데이터 이름만 바꾸면 불러오는 방법은 똑같습니다. 옷 이름은 `names[정답번호]`로 꺼냅니다.
```python
from tensorflow.keras.datasets import fashion_mnist                         # 옷 사진 데이터
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()            # 불러오기 (mnist 와 똑같음)
names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',                 # 0~4번 옷 이름
         'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']                  # 5~9번 옷 이름
```

# 2. 신경망 모델 만들기
`입력(28x28) → Flatten(784) → Dense(128, relu) → Dense(10, softmax)`
| 용어 | 뜻 |
|---|---|
| 층(layer) | 신경망을 이루는 한 단계. 층을 **쌓아서** 모델을 만든다 |
| 노드 | 층 안의 계산 단위. `Dense(128)` = 노드 128개 |
| 은닉층 | 입력층과 출력층 사이의 층 |
| `relu` | 음수는 0, 양수는 그대로 → 은닉층에 사용 |
| `softmax` | 출력값을 **합이 1인 확률**로 바꿈 → 출력층에 사용 |
| 출력층 노드 수 | **분류할 종류의 수** (0~9 → 10개) |

In [ ]:
# 신경망 모델 만들기: 입력 → 층 쌓기 → 출력
model = Sequential()                                        # 층을 쌓을 빈 모델 틀
model.add(Input(shape=(28, 28)))                            # 입력층: 28x28 사진 한 장씩 받기
model.add(Flatten())                                        # 28x28 이미지를 한 줄(784개)로 펼치기
model.add(Dense(128, activation='relu'))                    # 노드 128개짜리 은닉층
model.add(Dense(10, activation='softmax'))                  # 출력층: 숫자 0~9 각각일 확률 (10종류라 노드 10개)

In [ ]:
# 층마다 출력 모양과 파라미터 수 확인
model.summary()                                             # flatten 784 / dense 100,480 / dense_1 1,290 / 합계 101,770

#### 🔁 변형 ⑤: 노드 수 바꾸기 (64 / 256)
`Dense(128, ...)`의 숫자만 바꾸면 됩니다. 64개 → 50,890 / 256개 → 203,530

In [ ]:
model64 = Sequential()                                      # 빈 모델 틀
model64.add(Input(shape=(28, 28)))                          # 입력층
model64.add(Flatten())                                      # 한 줄로 펼치기 (784)
model64.add(Dense(64, activation='relu'))                   # 노드 64개짜리 은닉층
model64.add(Dense(10, activation='softmax'))                # 출력층
model64.summary()                                           # 합계 50,890

#### 🔁 변형 ⑥: 은닉층 2개 쌓기 (128 → 64)

In [ ]:
deep_model = Sequential()                                   # 빈 모델 틀
deep_model.add(Input(shape=(28, 28)))                       # 입력층
deep_model.add(Flatten())                                   # 한 줄로 펼치기 (784)
deep_model.add(Dense(128, activation='relu'))               # 은닉층 1: 노드 128개
deep_model.add(Dense(64, activation='relu'))                # 은닉층 2: 노드 64개 (층을 하나 더 쌓음)
deep_model.add(Dense(10, activation='softmax'))             # 출력층
deep_model.summary()                                        # 합계 109,386 (64 층: (128 + 1) x 64 = 8,256)

#### 🔁 변형 ⑦: 분류할 종류 수가 다르면?
출력층 노드 수 = **분류할 종류 수**. 예) 3종류 → `Dense(3, activation='softmax')` (정답 번호는 0, 1, 2)

# 3. 학습시키기
- `compile`: 학습 방법 정하기 / `fit`: 실제로 학습
- `epochs`: 학습용 데이터 **전체를 몇 번 반복**할지
- 한 번에 32장씩(batch) → 60000 ÷ 32 = **1875번**이 1 epoch

In [ ]:
# 학습 방법 정하기: 틀린 정도(loss)를 줄이도록 학습하고 정확도를 함께 기록
model.compile(optimizer='adam',                             # 모델의 값을 고쳐주는 방법 (adam)
              loss='sparse_categorical_crossentropy',       # 틀린 정도 계산 방법 (정답이 0~9 번호일 때)
              metrics=['accuracy'])                         # 학습하면서 정확도도 기록

In [ ]:
# 학습용 데이터 전체를 5번 반복해서 학습
history = model.fit(X_train, y_train,                       # 학습용 사진, 학습용 정답
                    epochs=5,                               # 전체를 5번 반복 (문제에서 말한 숫자로 바꾸기)
                    validation_data=(X_test, y_test))       # 매 epoch 끝에 테스트용으로 점수 확인 (val_)
# history.history 안에 epoch 마다 'loss', 'accuracy', 'val_loss', 'val_accuracy' 가 기록됨

- `loss` / `accuracy`: **학습용** 데이터 점수
- `val_loss` / `val_accuracy`: **테스트용**(처음 보는) 데이터 점수

숫자보다 **경향**을 보세요: loss는 내려가고 accuracy는 올라가나요?

# 4. 학습 곡선 그리고 읽기

In [ ]:
# loss(틀린 정도) 곡선: 학습용과 테스트용
plt.plot(history.history['loss'], label='train')            # 학습용 loss (x축 = epoch 번호 0, 1, 2)
plt.plot(history.history['val_loss'], label='test')         # 테스트용 loss (이름이 val_ 로 시작)
plt.xlabel('epoch')                                         # x축 이름
plt.ylabel('loss')                                          # y축 이름
plt.legend()                                                # label 로 붙인 이름표 표시
plt.show()                                                  # 보여주기

In [ ]:
# accuracy(정확도) 곡선: 학습용과 테스트용
plt.plot(history.history['accuracy'], label='train')        # 학습용 정확도
plt.plot(history.history['val_accuracy'], label='test')     # 테스트용 정확도
plt.xlabel('epoch')                                         # x축 이름
plt.ylabel('accuracy')                                      # y축 이름
plt.legend()                                                # 이름표 표시
plt.show()                                                  # 보여주기

**곡선 읽는 법**
- loss는 내려가고 accuracy는 올라가면 → 학습이 잘 되고 있음
- 학습용(train) 선이 테스트용(test) 선보다 좋으면 → 이미 본 문제라 더 잘 맞힘 (정상)
- train 은 계속 좋아지는데 test 는 오히려 나빠지면 → **과대적합** (학습용을 외워버림)

# 5. 노드 수를 줄이면?
노드를 128개에서 **16**개로 줄여 다시 학습하고 테스트용 정확도를 비교합니다.

In [ ]:
# 노드 16개짜리 작은 모델 만들고 학습시키기
small_model = Sequential()                                  # 빈 모델 틀
small_model.add(Input(shape=(28, 28)))                      # 입력층
small_model.add(Flatten())                                  # 한 줄로 펼치기 (784)
small_model.add(Dense(16, activation='relu'))               # 노드 16개짜리 은닉층 (128 -> 16)
small_model.add(Dense(10, activation='softmax'))            # 출력층
small_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])     # 학습 방법 (똑같이)
small_history = small_model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))          # 같은 조건(5번)으로 학습

In [ ]:
# 마지막 epoch의 테스트용 정확도 비교
print('노드 128개:', round(history.history['val_accuracy'][-1], 4))        # [-1] = 마지막 epoch, round(값, 4) = 소수 넷째 자리
print('노드 16개:', round(small_history.history['val_accuracy'][-1], 4))   # 작은 모델의 마지막 테스트 정확도
print('128개 파라미터:', model.count_params())                             # 101770
print('16개 파라미터:', small_model.count_params())                        # 12730

목표 결과 (숫자는 조금 달라도 됩니다): 노드 128개 **0.97xx** / 노드 16개 **0.94xx**

#### 🔁 변형 ⑧: 두 모델의 테스트 정확도 곡선 겹쳐 그리기

In [ ]:
plt.plot(history.history['val_accuracy'], label='128 nodes')        # 128개 모델 테스트 정확도
plt.plot(small_history.history['val_accuracy'], label='16 nodes')   # 16개 모델 테스트 정확도
plt.xlabel('epoch')                                                 # x축 이름
plt.ylabel('val_accuracy')                                          # y축 이름
plt.legend()                                                        # 이름표 표시
plt.show()                                                          # 보여주기

#### 🔁 변형 ⑨: loss 비교 / 첫 epoch 값
`[...]` 안의 이름만 바꾸면 됩니다: `'loss'`, `'accuracy'`, `'val_loss'`, `'val_accuracy'`
```python
print('128개 테스트 loss:', round(history.history['val_loss'][-1], 4))   # 테스트 loss (작을수록 좋음)
print('128개 첫 epoch:', round(history.history['val_accuracy'][0], 4))  # [0] = 첫 번째 epoch
```

#### 🔁 변형 ⑩: AI의 답 확인하기 (predict)
```python
pred = model.predict(X_test[:10], verbose=0).argmax(axis=1)     # 앞 10장의 확률 중 가장 큰 번호 = AI의 답
print(pred, y_test[:10])                                        # AI의 답과 정답 비교
print('맞힌 개수:', (pred == y_test[:10]).sum(), '/ 10')        # 같으면 True(1) -> 더하면 맞힌 개수
```

# 6. 개념 이해 예상 문제 + 답

**Q1. 층과 노드란?**
신경망은 **층을 쌓아서** 만든 모델이고, 노드는 층 안의 계산 단위이다. `Dense(128)`은 노드가 128개인 층이다.

**Q2. Flatten의 역할은? 출력 크기는?**
28×28 사진을 **한 줄로 펼쳐** Dense 층에 넣을 수 있게 만든다. 28 × 28 = **784**. 파라미터는 0개.

**Q3. Dense(128)의 파라미터가 100,480개인 이유는?**
들어오는 값 784개와 더하는 값 1개를 합한 785개가 노드 128개마다 있으므로 (784 + 1) × 128 = **100,480**.

**Q4. 출력층이 Dense(10, softmax)인 이유는?**
0~9 숫자 **10종류**를 맞히므로 노드 10개. `softmax`는 출력을 **합이 1인 확률**로 바꾸고, 가장 큰 확률의 번호가 AI의 답이다.

**Q5. relu는?**
음수는 0으로, 양수는 그대로 내보내는 함수로 은닉층에 쓴다.

**Q6. 픽셀값을 255로 나누는 이유는?**
0~255를 **0~1**로 맞춰서(정규화) 숫자 크기를 고르게 하면 학습이 안정적이다. 표 데이터에서 `MinMaxScaler`로 맞춘 것과 같은 이유. 학습용·테스트용 **둘 다** 나눈다.

**Q7. epoch이란? 5 epoch 은 몇 번 고치나요?**
학습용 데이터 **전체를 한 바퀴** 학습하는 것. 32장씩 묶어 1 epoch 에 60000 ÷ 32 = 1875번, 5 epoch 은 **9375번** 고친다.

**Q8. loss와 accuracy, val_ 의 차이는?**
loss는 틀린 정도(작을수록 좋음), accuracy는 맞힌 비율(클수록 좋음). `val_`이 붙은 값은 **테스트용**(처음 보는) 데이터 점수이고, 붙지 않은 값은 **학습용** 데이터 점수이다.

**Q9. 학습 곡선에서 학습용 선이 테스트용 선보다 좋은 이유는?**
학습용 사진은 모델이 여러 번 본 문제이고, 테스트용은 처음 보는 사진이기 때문이다.

**Q10. cv2.resize(img, (40, 20))의 결과 shape는?**
**(20, 40)**. resize는 (너비, 높이), shape는 (높이, 너비)라서 순서가 반대이다.

# 7. 논술형 예상 문제 + 모범 답안
**쓰는 순서: 결과(숫자) → 해석(왜) → 결론**. ⚠ 정확도는 **내 실행 결과 숫자로 바꿔 쓰세요.**

---
### 논술 1. 노드 수(128개 vs 16개)에 따른 결과를 비교하고 해석하시오.
**결과**: 노드 128개 모델의 테스트 정확도는 약 **0.97**, 노드 16개 모델은 약 **0.94**로 128개가 약 3%p 높았다.

**해석**: 노드 수가 많을수록 파라미터(가중치)가 많아져(**101,770개 vs 12,730개**) 손글씨의 더 다양한 특징을 표현할 수 있으므로 정확도가 높다. 하지만 파라미터가 약 8배 늘어난 것에 비해 정확도는 3%p 정도만 올랐고, 계산 시간과 메모리는 더 든다.

**결론**: 노드가 많을수록 대체로 정확도가 높지만, 늘어나는 계산량에 비해 효과가 점점 작아지므로 데이터에 맞는 적당한 크기를 고르는 것이 중요하다.

---
### 논술 2. 학습 곡선(loss, accuracy)을 보고 학습 과정을 해석하시오.
**결과**: epoch이 지날수록 학습용과 테스트용 loss는 모두 **내려가고**, accuracy는 모두 **올라갔다**. 마지막 epoch에서 학습용 정확도(약 **0.99**)가 테스트용 정확도(약 **0.97**)보다 조금 높았다.

**해석**: loss가 줄고 정확도가 오른 것은 모델이 학습을 반복하며 틀린 정도를 줄이는 방향으로 가중치를 고쳐 나갔기 때문이다. 학습용 점수가 더 높은 이유는 학습용 사진은 여러 번 본 문제이고 테스트용은 처음 보는 사진이기 때문이다. 두 선의 차이가 작으므로 외우기만 한 것이 아니라 처음 보는 사진도 잘 맞히는 **일반화**가 되었다.

---
### 논술 3. epoch을 5에서 50으로 늘리면 항상 더 좋아질까? 이유를 쓰시오.
**답**: 항상 좋아지지는 않는다. epoch을 늘리면 학습용 loss는 계속 줄어들지만, 어느 순간부터 테스트용 loss는 줄지 않거나 오히려 **올라갈 수 있다**. 모델이 학습용 사진의 세세한 부분까지 **외워버리는 과대적합** 때문이다. 그래서 테스트용(val_) 선이 더 이상 좋아지지 않는 지점에서 멈추는 것이 좋고, 학습 시간도 더 오래 걸린다.

---
### 논술 4. 은닉층을 하나 더 쌓으면(128 → 64) 어떻게 될지 쓰시오.
**답**: 파라미터가 101,770개에서 109,386개로 늘고, 앞 층이 찾은 특징을 한 번 더 조합할 수 있어 더 복잡한 특징을 표현할 수 있다. 정확도가 조금 오를 수 있지만 계산이 늘고, 데이터가 적거나 epoch이 많으면 과대적합이 생길 수 있다. 실제 효과는 테스트용 정확도로 확인해야 한다.